# Protocol 3 — fMRI Anticipation vs. Experience

Simulates Pred 3.A–3.D from Protocol 3 (*fMRI Anticipation vs. Experience Study: Dissociating Somatic Marker
Retrieval from Prediction Error*), updated to **protocol specification v1.1.0**:

* **Pred 3.A** — vmPFC anticipation BOLD is modulated by expected value (EV; p < 0.05 FWE) but does not track
  outcome-locked SCR (r < 0.20). *Falsified*: EV effect p > 0.10, or vmPFC–SCR r > 0.35.
* **Pred 3.B** — vmPFC→**aINS** PPI is higher in the anticipation foreperiod than in the outcome window
  (p < 0.05 FWE, Δr ≥ 0.30), while vmPFC→pIC stays flat (BF₀₁ ≥ 6). This is the precision-gain versus
  error-magnitude dissociation. *Falsified*: pIC coupling rises, or the aINS/pIC dissociation fails.
* **Pred 3.C** — vmPFC tracks EV (valence) but not the orthogonal visual cue contrast.
  *Falsified*: no EV effect, or vmPFC scales with sensory contrast.
* **Pred 3.D** — A 0 ms foreperiod collapses vmPFC→aINS coupling: long > 0 ms (p < 0.05, d > 0.4).

> All data are synthetic. Effect magnitudes are illustrative generative choices.
>
> **EP-3 | Paper 1 | Empirical | Depends on: no HEP dependency (insula interpretation follows Pred 0.C) |
> Spec v1.1.0, pre-collection | N enrolled = 40, analysable = 36**

### What changed from v1.0

| Item | v1.0 notebook | v1.1 specification |
|---|---|---|
| N | 36 target / 26 minimum | 40 enrolled, 36 analysable (d = 0.5, α = 0.05, power 0.80) |
| Trials | not modeled | 2 runs × 100 = 200 trials; risk × foreperiod 25 per run; ≥ 4 EV levels; orthogonal cue contrast |
| Foreperiod | 2000–4000 ms vs 0 ms | unchanged; outcome 2000 ms, ITI 6–10 s (≥ 8 s after shock) |
| PPI target | vmPFC–pIC plotted for 3.B/3.D | vmPFC→aINS (precision gain) vs vmPFC→pIC (flat control) |
| Pred 3.A | EV effect only | EV effect **and** vmPFC–SCR r < 0.20 |
| Gate | — | Identifiability gate: normalized cross-information ≤ 0.1, else β_SM/Πⁱ claims exploratory |
| κ, α, β | 100, 0.3, 0.7 | κ = `null`; no legacy α/β defaults |

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy import integrate
from scipy.stats import pearsonr, ttest_1samp, ttest_rel

In [ ]:
# Load the v1.1 protocol specification so every threshold below is read from
# the registered JSON rather than retyped by hand.
def load_protocol(filename):
    for root in (Path.cwd(), *Path.cwd().parents):
        path = root / "protocols" / filename
        if path.exists():
            return json.loads(path.read_text(encoding="utf-8"))
    raise FileNotFoundError(f"protocols/{filename} not found above {Path.cwd()}")


PROTO = load_protocol("protocol_3_anticipation_fmri.json")
assert PROTO["version"] == "1.1.0", PROTO["version"]
PRED = {p["id"]: p for p in PROTO["sub_predictions"]}
print(f"{PROTO['protocol_id']}  spec v{PROTO['version']}  status={PROTO['study_status']}")
for pid, p in PRED.items():
    print(f"  {pid:<9} [{p['status']}]  {p['claim'][:78]}")


def verdict(confirmed, falsified=False):
    """v1.1 three-way decision: failing to confirm is not falsification."""
    if confirmed:
        return "CONFIRMED (synthetic)"
    if falsified:
        return "FALSIFIED (synthetic)"
    return "INDETERMINATE (not confirmed; falsification rule not met)"

In [ ]:
def jzs_bf10(t, n, r=0.707):
    """JZS Bayes factor for a one-sample / paired t-test (Rouder et al., 2009).

    Cauchy prior scale r = 0.707, as registered for the v1.1 Bayesian tests.
    The likelihood ratio is formed inside the integral to avoid underflow.
    """
    nu = n - 1

    def integrand(g):
        ratio = ((1 + t**2 / ((1 + n * g * r**2) * nu)) / (1 + t**2 / nu)) ** (-(nu + 1) / 2)
        prior = (2 * np.pi) ** -0.5 * g**-1.5 * np.exp(-1 / (2 * g))
        return (1 + n * g * r**2) ** -0.5 * ratio * prior

    return float(integrate.quad(integrand, 0, np.inf, limit=200)[0])

## 1 — Protocol 3 design (v1.1)

In [ ]:
N_ENROLLED = PROTO["participants"]["n_target"]              # 40
N_ANALYSABLE = PROTO["participants"]["n_analyzable_target"] # 36
BLOCKS = PROTO["blocks"]
TRIALS = PROTO["trial_structure"]
N_RUNS = BLOCKS["experimental_blocks"]                      # 2
N_TRIALS = BLOCKS["total_experimental_trials"]              # 200
CELL = BLOCKS["risk_by_foreperiod_trials_per_run"]          # 25
LONG_FP = TRIALS["foreperiod_ms"][1]                        # [2000, 4000]
EV_LEVELS = np.array([-1.5, -0.5, 0.5, 1.5])                # ≥ 4 EV levels (standardized)
CONTRAST_LEVELS = np.array([-1.0, 1.0])                     # orthogonal cue contrast
assert N_RUNS * 2 * 2 * CELL == N_TRIALS                    # risk (2) × foreperiod (2) × 25 × 2 runs

GATE = PROTO["identifiability_gate"]
print(f"N enrolled {N_ENROLLED}, analysable {N_ANALYSABLE}; {N_RUNS} runs × {N_TRIALS // N_RUNS} trials")
print(f"Foreperiods: 0 ms vs {LONG_FP[0]}–{LONG_FP[1]} ms; outcome {TRIALS['outcome_ms']} ms; "
      f"ITI {TRIALS['iti_ms'][0]}–{TRIALS['iti_ms'][1]} ms")
print("Identifiability gate:", GATE["normalized_cross_information_maximum"], "|", GATE["failure"])
rng = np.random.default_rng(5)

## 2 — Simulate participants

Each participant contributes trial-level anticipation-window vmPFC estimates (EV × cue contrast design),
outcome-locked SCR, and PPI coefficients for vmPFC→aINS and vmPFC→pIC in each window and foreperiod.
Four enrolled participants are excluded (motion / missing runs) to reach the analysable N.

In [ ]:
def simulate_participant(rng):
    ev = rng.choice(EV_LEVELS, N_TRIALS)
    contrast = rng.choice(CONTRAST_LEVELS, N_TRIALS)
    b_ev = rng.normal(0.30, 0.15)                            # participant EV sensitivity
    vmpfc = b_ev * ev + 0.0 * contrast + rng.normal(0, 1.0, N_TRIALS)
    scr_outcome = rng.normal(0, 1, N_TRIALS) + 0.05 * vmpfc  # outcome-locked arousal, weakly related
    slope_ev, slope_con = np.linalg.lstsq(np.column_stack([np.ones(N_TRIALS), ev, contrast]), vmpfc,
                                          rcond=None)[0][1:]
    r_scr = pearsonr(vmpfc, scr_outcome)[0]
    ppi = dict(
        ains_antic_long=rng.normal(0.44, 0.13), ains_antic_none=rng.normal(0.10, 0.12),
        ains_outcome=rng.normal(0.11, 0.11),
        pic_antic=rng.normal(0.12, 0.10), pic_outcome=rng.normal(0.12, 0.10),
    )
    return dict(slope_ev=slope_ev, slope_con=slope_con, r_scr=r_scr, **ppi)


enrolled = [simulate_participant(rng) for _ in range(N_ENROLLED)]
excluded = rng.choice(N_ENROLLED, N_ENROLLED - N_ANALYSABLE, replace=False)
data = {k: np.array([p[k] for i, p in enumerate(enrolled) if i not in excluded]) for k in enrolled[0]}
n = len(data["slope_ev"])
print(f"Analysable participants: {n}")

## 3 — Pred 3.A and 3.C: vmPFC tracks EV, not sensory contrast or outcome SCR

In [ ]:
t_ev, p_ev = ttest_1samp(data["slope_ev"], 0.0)
t_con, p_con = ttest_1samp(data["slope_con"], 0.0)
r_scr = np.tanh(np.arctanh(data["r_scr"]).mean())

confirmed_3A = p_ev < 0.05 and r_scr < 0.20
falsified_3A = p_ev > 0.10 or r_scr > 0.35
bf01_con = 1 / jzs_bf10(t_con, n)
confirmed_3C = p_ev < 0.05 and p_con >= 0.05
falsified_3C = p_ev > 0.10 or (p_con < 0.05 and data["slope_con"].mean() > 0)

print(f"vmPFC EV slope = {data['slope_ev'].mean():.3f} (t({n - 1}) = {t_ev:.2f}, p = {p_ev:.2e})")
print(f"vmPFC contrast slope = {data['slope_con'].mean():+.3f} (p = {p_con:.2f}, BF₀₁ = {bf01_con:.2f})")
print(f"Pooled vmPFC–outcome SCR r = {r_scr:.3f}")
print(f"Pred 3.A → {verdict(confirmed_3A, falsified_3A)}")
print(f"Pred 3.C → {verdict(confirmed_3C, falsified_3C)}")

## 4 — Pred 3.B: vmPFC→aINS rises in anticipation; vmPFC→pIC stays flat

In [ ]:
d_ains = data["ains_antic_long"] - data["ains_outcome"]
d_pic = data["pic_antic"] - data["pic_outcome"]
t_ains, p_ains = ttest_rel(data["ains_antic_long"], data["ains_outcome"])
t_pic, p_pic = ttest_rel(data["pic_antic"], data["pic_outcome"])
bf01_pic = 1 / jzs_bf10(t_pic, n)
t_dis, p_dis = ttest_rel(d_ains, d_pic)

bf01_ceiling = 1 / jzs_bf10(0.0, n)  # largest BF₀₁ a paired JZS test can reach at this N
confirmed_3B = p_ains < 0.05 and d_ains.mean() >= 0.30 and bf01_pic >= 6
falsified_3B = (p_pic < 0.05 and d_pic.mean() > 0) or p_dis >= 0.05

print(f"vmPFC→aINS: anticipation − outcome Δ = {d_ains.mean():.3f} (p = {p_ains:.2e})")
print(f"vmPFC→pIC : anticipation − outcome Δ = {d_pic.mean():+.3f} (p = {p_pic:.2f}, BF₀₁ = {bf01_pic:.2f})")
print(f"aINS vs pIC dissociation: p = {p_dis:.2e}")
print(f"Design check: max attainable BF₀₁ (paired JZS, r = 0.707) at N = {n} is {bf01_ceiling:.2f}")
print(f"Pred 3.B → {verdict(confirmed_3B, falsified_3B)}")

> **Design flag.** With a default paired JZS test (Cauchy scale 0.707), even an observed t = 0 gives
> BF₀₁ ≈ 5.6 at N = 36 (≈ 5.9 at N = 40). The registered BF₀₁ ≥ 6 null conjunct for vmPFC→pIC is then
> unreachable, and Pred 3.B can at best be indeterminate. The prior, test family or N needs fixing in the
> analysis manifest before lock.

## 5 — Pred 3.D: Removing the foreperiod collapses vmPFC→aINS coupling

In [ ]:
diff_fp = data["ains_antic_long"] - data["ains_antic_none"]
t_fp, p_fp = ttest_rel(data["ains_antic_long"], data["ains_antic_none"])
d_fp = diff_fp.mean() / diff_fp.std(ddof=1)
confirmed_3D = p_fp < 0.05 and d_fp > 0.4
falsified_3D = p_fp >= 0.05 and 1 / jzs_bf10(t_fp, n) >= 3
print(f"vmPFC→aINS long FP − 0 ms = {diff_fp.mean():.3f} (t = {t_fp:.2f}, p = {p_fp:.2e}, d_z = {d_fp:.2f})")
print(f"Pred 3.D → {verdict(confirmed_3D, falsified_3D)}")

## 6 — Visualize Pred 3.A–3.D

In [ ]:
def bar(ax, values, labels, colors, title, ylabel):
    ax.bar(labels, [v.mean() for v in values], yerr=[v.std() / np.sqrt(n) for v in values], color=colors,
           alpha=0.85, edgecolor="white", width=0.5, capsize=5)
    ax.axhline(0, ls="--", lw=0.8, color="k", alpha=0.4)
    ax.set_title(title, fontsize=10)
    ax.set_ylabel(ylabel)
    ax.spines[["top", "right"]].set_visible(False)


fig, axes = plt.subplots(1, 4, figsize=(16, 4))
bar(axes[0], [data["slope_ev"], data["slope_con"]], ["EV\n(valence)", "Cue\ncontrast"],
    ["#2166ac", "#d6604d"], "Pred 3.A/3.C — vmPFC anticipation", "vmPFC β (a.u.)")
bar(axes[1], [data["ains_antic_long"], data["ains_outcome"]], ["Anticipation", "Outcome"],
    ["#2166ac", "#9966FF"], f"Pred 3.B — vmPFC→aINS\nΔ = {d_ains.mean():.2f}", "PPI coefficient")
bar(axes[2], [data["pic_antic"], data["pic_outcome"]], ["Anticipation", "Outcome"],
    ["#AAAAAA", "#CCCCCC"], f"Pred 3.B — vmPFC→pIC (flat)\nBF₀₁ = {bf01_pic:.1f}", "PPI coefficient")
bar(axes[3], [data["ains_antic_long"], data["ains_antic_none"]],
    [f"Long FP\n({LONG_FP[0]}–{LONG_FP[1]} ms)", "No FP\n(0 ms)"],
    ["#2166ac", "#AAAAAA"], f"Pred 3.D — foreperiod\nd_z = {d_fp:.2f}", "vmPFC→aINS PPI")
fig.suptitle(f"Protocol 3 — fMRI anticipation (Pred 3.A–3.D) · spec v1.1.0 · synthetic N = {n}", fontsize=12)
plt.tight_layout()
plt.show()

## 7 — APGI interpretation

$\hat M(c,a) = \gamma_V V(c,a) + \gamma_A A(c,a)$ acts on the **precision gain** before outcome:

$$\Pi^i_{\text{eff}} = \Pi^i_{\text{baseline}} \cdot e^{\beta_{SM} \hat M(c,a)}$$

The anticipation-window vmPFC signal indexes M̂ retrieval, not outcome-locked εⁱ (Pred 3.A). The
precision-gain pathway terminates in aINS, so vmPFC→aINS coupling rises with anticipation while vmPFC→pIC,
the sensory-intensity pathway, stays flat (Pred 3.B). Removing the foreperiod removes the retrieval window
(Pred 3.D).

β_SM and Πⁱ are not separately identifiable from behavior alone. If the identifiability gate fails
(normalized cross-information > 0.1), individual β_SM/Πⁱ claims become exploratory and only the
prospectively locked composite model remains confirmatory.